In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/zaidashraf/Quantum-Secured-Threat-Intelligence-Pipeline


In [2]:
import torch
import transformers

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

/opt/homebrew/Cellar/jupyterlab/4.4.3/libexec/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.12.0
Transformers version: 5.9.0
MPS available: True
CUDA available: False


In [3]:
from pathlib import Path

REPORT_PATH = Path(
    "../data/raw/sample_reports/testing/report_001.txt"
)

report_text = REPORT_PATH.read_text(
    encoding="utf-8"
)

print(report_text)

APT29 used PowerShell to execute malicious commands and exploit
CVE-2023-38831. The activity was mapped to T1059.001 and contacted
192.168.1.25 through https://malicious-example.com/payload.

The downloaded file had the SHA-256 hash
0123456789abcdef0123456789abcdef0123456789abcdef0123456789abcdef.


In [4]:
import importlib
import src.bert_ner

importlib.reload(src.bert_ner)

from src.bert_ner import TransformerNER

In [5]:
transformer_ner = TransformerNER()

Loading weights: 100%|█████████████████████| 138/138 [00:00<00:00, 8098.70it/s]


In [6]:
transformer_ner.model.config.id2label

{0: 'B-Indicator',
 1: 'B-Malware',
 2: 'B-Organization',
 3: 'B-System',
 4: 'B-Vulnerability',
 5: 'I-Indicator',
 6: 'I-Malware',
 7: 'I-Organization',
 8: 'I-System',
 9: 'I-Vulnerability',
 10: 'O'}

In [7]:
ml_entities = transformer_ner.extract(
    text=report_text,
    minimum_score=0.50,
)

ml_entities

[{'text': ' Power',
  'label': 'System',
  'score': 0.6877,
  'start': 10,
  'end': 16,
  'source': 'transformer'},
 {'text': 'Shell',
  'label': 'Malware',
  'score': 0.8664,
  'start': 16,
  'end': 21,
  'source': 'transformer'},
 {'text': ' execute',
  'label': 'Indicator',
  'score': 0.8055,
  'start': 24,
  'end': 32,
  'source': 'transformer'},
 {'text': ' malicious commands',
  'label': 'Indicator',
  'score': 0.8261,
  'start': 32,
  'end': 51,
  'source': 'transformer'},
 {'text': '\n',
  'label': 'Indicator',
  'score': 0.7369,
  'start': 63,
  'end': 64,
  'source': 'transformer'},
 {'text': 'CVE-',
  'label': 'Vulnerability',
  'score': 0.5532,
  'start': 64,
  'end': 68,
  'source': 'transformer'},
 {'text': '20',
  'label': 'Vulnerability',
  'score': 0.612,
  'start': 68,
  'end': 70,
  'source': 'transformer'},
 {'text': '.',
  'label': 'Vulnerability',
  'score': 0.8446,
  'start': 78,
  'end': 79,
  'source': 'transformer'},
 {'text': '10',
  'label': 'Indicator',
  '

In [8]:
import pandas as pd

ml_df = pd.DataFrame(ml_entities)

ml_df[
    ["text", "label", "score", "start", "end"]
].sort_values(
    by="start"
).reset_index(drop=True)

,text,label,score,start,end
0,Power,System,0.6877,10,16
1,Shell,Malware,0.8664,16,21
2,execute,Indicator,0.8055,24,32
3,malicious commands,Indicator,0.8261,32,51
4,\n,Indicator,0.7369,63,64
5,CVE-,Vulnerability,0.5532,64,68
6,20,Vulnerability,0.6120,68,70
7,.,Vulnerability,0.8446,78,79
8,10,Indicator,0.9914,108,110
9,59,Indicator,0.7186,110,112


In [9]:
ml_df["label"].value_counts()

label
Indicator        44
Vulnerability     3
System            1
Malware           1
Name: count, dtype: int64

In [10]:
ml_df.groupby("label")["score"].agg(
    ["count", "mean", "min", "max"]
)

,count,mean,min,max
label,,,,
Indicator,44,0.947723,0.5905,1.0000
Malware,1,0.866400,0.8664,0.8664
System,1,0.687700,0.6877,0.6877
Vulnerability,3,0.669933,0.5532,0.8446


In [11]:
ml_entities_high_confidence = transformer_ner.extract(
    text=report_text,
    minimum_score=0.80,
)

pd.DataFrame(ml_entities_high_confidence)

,text,label,score,start,end,source
0,Shell,Malware,0.8664,16,21,transformer
1,execute,Indicator,0.8055,24,32,transformer
2,malicious commands,Indicator,0.8261,32,51,transformer
3,.,Vulnerability,0.8446,78,79,transformer
4,10,Indicator,0.9914,108,110,transformer
5,\n,Indicator,0.9004,130,131,transformer
6,192,Indicator,1.0000,131,134,transformer
7,.,Indicator,1.0000,134,135,transformer
8,168,Indicator,1.0000,135,138,transformer
9,.,Indicator,1.0000,138,139,transformer


In [12]:
for entity in ml_entities_high_confidence:
    print(
        f"{entity['text']!r:<30} "
        f"{entity['label']:<20} "
        f"{entity['score']:.3f}"
    )

'Shell'                        Malware              0.866
' execute'                     Indicator            0.805
' malicious commands'          Indicator            0.826
'.'                            Vulnerability        0.845
'10'                           Indicator            0.991
'\n'                           Indicator            0.900
'192'                          Indicator            1.000
'.'                            Indicator            1.000
'168'                          Indicator            1.000
'.'                            Indicator            1.000
'1'                            Indicator            0.998
'.'                            Indicator            1.000
'25'                           Indicator            1.000
' https'                       Indicator            0.997
'://'                          Indicator            0.992
'mal'                          Indicator            0.972
'icious-example.com/payload.\n' Indicator            0.917
' the'       